In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import xtrack as xt
import xpart as xp
import xobjects as xo
import xplt
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex'])
import pickle
from datetime import datetime
from models import *

from scipy import stats
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

torch.set_default_dtype(torch.float64)

In [ ]:
# Define accelerator

pi = np.pi
lbend = 3

# Create an environment
env = xt.Environment()

env['k1f'] = 0.1
env['k1d'] = -0.7

# Build a simple ring
line = env.new_line(components=[
    env.new('mqf.1', xt.Quadrupole, length=0.3, k1='k1f'),
    env.new('d1.1',  xt.Drift, length=1),
    env.new('mb1.1', xt.Bend, length=lbend, k0=pi / 2 / lbend, h=pi / 2 / lbend),
    env.new('d2.11',  xt.Drift, length=0.3),
    env.new('mof.1', xt.Octupole, length=0.4, k3=0.1), # octupolar perturbation
    env.new('d2.12',  xt.Drift, length=0.3),

    env.new('mqd.1', xt.Quadrupole, length=0.3, k1='k1d'),
    env.new('d3.1',  xt.Drift, length=1),
    env.new('mb2.1', xt.Bend, length=lbend, k0=pi / 2 / lbend, h=pi / 2 / lbend),
    env.new('d4.1',  xt.Drift, length=1),

    env.new('mqf.2', xt.Quadrupole, length=0.3, k1='k1f'),
    env.new('d1.2',  xt.Drift, length=1),
    env.new('mb1.2', xt.Bend, length=lbend, k0=pi / 2 / lbend, h=pi / 2 / lbend),
    env.new('d2.2',  xt.Drift, length=1),

    env.new('mqd.2', xt.Quadrupole, length=0.3, k1='k1d'),
    env.new('d3.2',  xt.Drift, length=1),
    env.new('mb2.2', xt.Bend, length=lbend, k0=pi / 2 / lbend, h=pi / 2 / lbend),
    env.new('d4.2',  xt.Drift, length=1),
])

# Define reference particle
line.particle_ref = xt.Particles(p0c=1.2e9, mass0=xt.PROTON_MASS_EV)

In [ ]:
# survey
fig, ax = plt.subplots(1, figsize=(4.5, 4.5))

survey = line.survey()
plot = xplt.FloorPlot(survey, element_width=1.6, ax=ax)
plot.legend(fontsize=11)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.set_xlabel('Z [m]', size=14)
ax.set_ylabel('X [m]', size=14)

plt.savefig('figures/accelerator_schematic.pdf')

In [ ]:
initial_k1f = line['k1f']
initial_k1d = line['k1d']

In [ ]:
# track particles and pickle results

np.random.seed(144)
line.discard_tracker()
line.build_tracker(_context=xo.ContextCpu(omp_num_threads='auto'))

#parameters = [0.5095, 0.61, 0.71]
parameters = [0.57, 0.68]


n_turns = 1000
record_every = 1
num_particles = 10000
y = np.zeros((num_particles, n_turns//record_every + 1))
py = np.zeros((num_particles, n_turns//record_every + 1))
k1f = np.zeros((num_particles, n_turns//record_every + 1))
k1d = np.zeros((num_particles, n_turns//record_every + 1))

data_y = []
data_py = []
start_time = datetime.now()
for j in range(len(parameters)):
    line['k1f'] = initial_k1f
    line['k1d'] = initial_k1d

    line.match(
            method='4d',
            vary=[
                xt.VaryList(['k1f'], step=1e-8, tag='quad'),
                xt.VaryList(['k1d'], step=1e-8, tag='quad'),
            ],
            targets = [
                xt.TargetSet(qx=1.36, tol=1e-6, tag='tune'),
                xt.TargetSet(qy=parameters[j], tol=1e-6, tag='tune'),
            ])

    final_k1f = line['k1f']
    final_k1d = line['k1d']

    line['k1f'] = initial_k1f
    line['k1d'] = initial_k1d
    
    y_in_sigmas, py_in_sigmas = xp.generate_2D_gaussian(num_particles)

    particles = line.build_particles(
                y_norm=y_in_sigmas, py_norm=py_in_sigmas,
                nemitt_y=13e-6, method = "4d")

    y[:, 0] = particles.y
    py[:, 0] = particles.py
    k1f[:, 0] = line['k1f']
    k1d[:, 0] = line['k1d']
    for i in range(n_turns):
        if (i+1) % record_every == 0:
            k1f[:, i//record_every] = line['k1f']
            k1d[:, i//record_every] = line['k1d']

        line['k1f'] = initial_k1f + (final_k1f - initial_k1f) * i / n_turns
        line['k1d'] = initial_k1d + (final_k1d - initial_k1d) * i / n_turns
        line.track(particles, num_turns=1, freeze_longitudinal=True)

        if (i+1) % record_every == 0:
            y[:, (i+1)//record_every] = particles.y
            py[:, (i+1)//record_every] = particles.py
            k1f[:, (i+1)//record_every] = line['k1f']
            k1d[:, (i+1)//record_every] = line['k1d']

    with open(f'data/simple_line_10k_perturbed_{parameters[j]}.pkl', 'wb') as f:
        pickle.dump((y, py, k1f, k1d), f)

end_time = datetime.now()

# 0:17:53.713859
print(end_time - start_time)